[ToXCL Github](https://github.com/NhatHoang2002/ToXCL)

[ToXCL Paper](https://aclanthology.org/2024.naacl-long.359.pdf)

NOTE

This file (basline_runer.ipynb) was ran in Kaggle to use the provided P1000 GPU

The data and other .py files were uploaded as inputs, their paths are:

* /kaggle/input/datasets/veerhanglitch/toxcl-data
    * /kaggle/input/datasets/veerhanglitch/toxcl-data/IHC_train.csv
    * /kaggle/input/datasets/veerhanglitch/toxcl-data/IHC_valid.csv
    * /kaggle/input/datasets/veerhanglitch/toxcl-data/SBIC_test.csv
    * /kaggle/input/datasets/veerhanglitch/toxcl-data/SBIC_train.csv
    * /kaggle/input/datasets/veerhanglitch/toxcl-data/SBIC_valid.csv
    * /kaggle/input/datasets/veerhanglitch/toxcl-data/TG_train.csv    (unused)
    * /kaggle/input/datasets/veerhanglitch/toxcl-data/TG_valid.csv    (unused)
* /kaggle/input/datasets/veerhanglitch/toxcl-files
    * /kaggle/input/datasets/veerhanglitch/toxcl-files/baselines/train_decoder_arch.py
    * /kaggle/input/datasets/veerhanglitch/toxcl-files/baselines/train_encoder_arch.py

Wandb was used to act as result storage to remove the problem of kaggle output storage limits.

Paper Mentions:


'''
**Baselines** We compare the performance of our TG model with three baseline models: 
1. **BERT (Devlin et al., 2019):** an encoder-based model
2. **GPT-2 (Radford et al., 2019):** a decoder-only model
3. **BART (Lewis et al., 2020):** an encoder-decoder model. BERT is widely used for multi-label classification tasks while both GPT-2 and BART have demonstrated remarkable performance in text
'''

In [ ]:
# installing the requriments with ver as provided in requirments.txt in GitHub

!pip install accelerate==0.33.0 bert_score==0.3.13 evaluate==0.4.1 openai rouge-score==0.1.2 scikit-learn==1.3.2 tokenizers==0.19.1 torch==2.2.1 transformers==4.44.2 wandb

In [ ]:
# kaggle env check

import torch
import os
import subprocess

if torch.cuda.is_available():
    device_name = torch.cuda.get_device_name(0)
    print(f"GPU DETECTED: {device_name}")
    !nvidia-smi --query-gpu=memory.total --format=csv
else:
    print("ERROR: No GPU found")

try:
    import socket
    socket.create_connection(("www.google.com", 80))
    print("INTERNET: Connected")
except OSError:
    print("ERROR: No Internet")

from kaggle_secrets import UserSecretsClient
try:
    UserSecretsClient().get_secret("WANDB_API_KEY")
    print("WANDB SECRET: Found")
except:
    print("ERROR: WANDB_API_KEY not found")

In [ ]:
# req imports

import transformers
from transformers import TrainingArguments
import shutil
import wandb
import os
from kaggle_secrets import UserSecretsClient

In [ ]:
# wandb login

user_secrets = UserSecretsClient()
wandb.login(key=user_secrets.get_secret("WANDB_API_KEY"))

In [ ]:
BASELINE 1: Encoder

In [ ]:
# command given in github

# # `model_checkpoint` used in paper: GroNLP/hateBERT, bert-base-uncased, google/electra-base-discriminator, roberta-base
# python -m baselines.train_encoder_arch \
#     --model_name {model_checkpoint} \
#     --output_dir {output_dir} \
#     --dataset_name {IHC | SBIC}

In [ ]:
# basline 1 function to reduce cluter due to stuff like try except and wandb storing

def basline_encoder(model_name, train_file, valid_file, test_file, output_dir, name):
    try:
        print(f"Starting {name}")
        !python /kaggle/input/datasets/veerhanglitch/toxcl-files/baselines/train_encoder_arch.py \
            --model_name {model_name} \
            --train_file {train_file} \
            --valid_file {valid_file} \
            --test_file {test_file} \
            --output_dir {output_dir}
        
        zip_path = f"/kaggle/working/{name}_results"
        shutil.make_archive(zip_path, 'zip', output_dir)
        
        run = wandb.init(project="toxcl-runs", name=name)
        artifact = wandb.Artifact(name=name, type="model_output")
        artifact.add_file(f"{zip_path}.zip")
        run.log_artifact(artifact)
        run.finish()
        print(f"{name} completed and uploaded.")

    except Exception as e:
        # If it fails, tell W&B so you know exactly what happened
        run = wandb.init(project="toxcl-runs", name=f"FAILED_{name}")
        wandb.log({"error_message": str(e)})
        run.finish()
        print(f"{name} FAILED. Error logged to W&B.")

    finally:
        # ALWAYS clean up disk, even if the training crashed mid-way
        # This prevents "Disk Quota Exceeded" from killing the REST of your night
        if os.path.exists(output_dir):
            shutil.rmtree(output_dir)
        if os.path.exists(f"{zip_path}.zip"):
            os.remove(f"{zip_path}.zip")
        print(f"Disk cleaned.")

In [ ]:
# IHC hateBERT

# !python /kaggle/input/datasets/veerhanglitch/toxcl-files/baselines/train_encoder_arch.py \
#     --model_name GroNLP/hateBERT \
#     --train_file /kaggle/input/datasets/veerhanglitch/toxcl-data/IHC_train.csv \
#     --valid_file /kaggle/input/datasets/veerhanglitch/toxcl-data/IHC_valid.csv \
#     --test_file /kaggle/input/datasets/veerhanglitch/toxcl-data/IHC_valid.csv \
#     --output_dir /kaggle/working/outputs/encode/IHC/GroNLP-hateBERT

basline_encoder('GroNLP/hateBERT', '/kaggle/input/datasets/veerhanglitch/toxcl-data/IHC_train.csv', '/kaggle/input/datasets/veerhanglitch/toxcl-data/IHC_valid.csv', '/kaggle/input/datasets/veerhanglitch/toxcl-data/IHC_valid.csv', '/kaggle/working/outputs/encode/IHC/GroNLP-hateBERT', 'encode_IHC_GroNLP-hateBERT')


In [ ]:
# IHC BERT

# !python /kaggle/input/datasets/veerhanglitch/toxcl-files/baselines/train_encoder_arch.py \
#     --model_name bert-base-uncased \
#     --train_file /kaggle/input/datasets/veerhanglitch/toxcl-data/IHC_train.csv \
#     --valid_file /kaggle/input/datasets/veerhanglitch/toxcl-data/IHC_valid.csv \
#     --test_file /kaggle/input/datasets/veerhanglitch/toxcl-data/IHC_valid.csv \
#     --output_dir /kaggle/working/outputs/encode/IHC/bert-base-uncased

basline_encoder('bert-base-uncased', '/kaggle/input/datasets/veerhanglitch/toxcl-data/IHC_train.csv', '/kaggle/input/datasets/veerhanglitch/toxcl-data/IHC_valid.csv', '/kaggle/input/datasets/veerhanglitch/toxcl-data/IHC_valid.csv', '/kaggle/working/outputs/encode/IHC/bert-base-uncased', 'encode_IHC_bert-base-uncased')


In [ ]:
# IHC ELECTRA

# the cmd stuff was required to avoid a ValueError caused by ELECTRA's non-contiguous attention tensors as they are incompatible with the modern default Safetensors serialization format.
# It fixees the prob by forcing a fallback to the standard PyTorch .bin format.

# !python /kaggle/input/datasets/veerhanglitch/toxcl-files/baselines/train_encoder_arch.py \
#     --model_name google/electra-base-discriminator \
#     --train_file /kaggle/input/datasets/veerhanglitch/toxcl-data/IHC_train.csv \
#     --valid_file /kaggle/input/datasets/veerhanglitch/toxcl-data/IHC_valid.csv \
#     --test_file /kaggle/input/datasets/veerhanglitch/toxcl-data/IHC_valid.csv \
#     --output_dir /kaggle/working/outputs/encode/IHC/encode_IHC_google-electra-base-discriminator


patch_logic = (
    "import transformers.utils.versions; "
    "transformers.utils.versions.require_version = lambda *a, **k: None; " # bypas the 'PackageNotFoundError'
    "from transformers import TrainingArguments; "
    "orig = TrainingArguments.__init__; "
    "TrainingArguments.__init__ = lambda self, *a, **k: orig(self, *a, **{**k, 'save_safetensors': False}); " # force .bin save
)

# try except used here also just in case !{cmd} gives an error
try:   
    cmd = (
        f"python -c \"{patch_logic} "
        f"import sys; sys.argv = ['', "
        f"'--model_name', 'google/electra-base-discriminator', "
        f"'--train_file', '/kaggle/input/datasets/veerhanglitch/toxcl-data/IHC_train.csv', "
        f"'--valid_file', '/kaggle/input/datasets/veerhanglitch/toxcl-data/IHC_valid.csv', "
        f"'--test_file', '/kaggle/input/datasets/veerhanglitch/toxcl-data/IHC_valid.csv', "
        f"'--output_dir', '/kaggle/working/outputs/encode/IHC/google-electra-base-discriminator']; "
        f"exec(open('/kaggle/input/datasets/veerhanglitch/toxcl-files/baselines/train_encoder_arch.py').read())\""
    )

    !{cmd}

    basline_encoder('google/electra-base-discriminator', '/kaggle/input/datasets/veerhanglitch/toxcl-data/IHC_train.csv', '/kaggle/input/datasets/veerhanglitch/toxcl-data/IHC_valid.csv', '/kaggle/input/datasets/veerhanglitch/toxcl-data/IHC_valid.csv', '/kaggle/working/outputs/encode/IHC/google-electra-base-discriminator', 'encode_IHC_google-electra-base-discriminator')

except Exception as e:
    print(f"Error: {e}")

In [ ]:
# IHC RoBERTa

# !python /kaggle/input/datasets/veerhanglitch/toxcl-files/baselines/train_encoder_arch.py \
#     --model_name roberta-base \
#     --train_file /kaggle/input/datasets/veerhanglitch/toxcl-data/IHC_train.csv \
#     --valid_file /kaggle/input/datasets/veerhanglitch/toxcl-data/IHC_valid.csv \
#     --test_file /kaggle/input/datasets/veerhanglitch/toxcl-data/IHC_valid.csv \
#     --output_dir /kaggle/working/outputs/encode/IHC/roberta-base

basline_encoder('roberta-base', '/kaggle/input/datasets/veerhanglitch/toxcl-data/IHC_train.csv', '/kaggle/input/datasets/veerhanglitch/toxcl-data/IHC_valid.csv', '/kaggle/input/datasets/veerhanglitch/toxcl-data/IHC_valid.csv', '/kaggle/working/outputs/encode/IHC/roberta-base', 'encode_IHC_roberta-base')

In [ ]:
# SBIC hateBERT

# !python /kaggle/input/datasets/veerhanglitch/toxcl-files/baselines/train_encoder_arch.py \
#     --model_name GroNLP/hateBERT \
#     --train_file /kaggle/input/datasets/veerhanglitch/toxcl-data/SBIC_train.csv \
#     --valid_file /kaggle/input/datasets/veerhanglitch/toxcl-data/SBIC_valid.csv \
#     --test_file /kaggle/input/datasets/veerhanglitch/toxcl-data/SBIC_test.csv \
#     --output_dir /kaggle/working/outputs/encode/SBIC/GroNLP-hateBERT

basline_encoder('GroNLP/hateBERT', '/kaggle/input/datasets/veerhanglitch/toxcl-data/SBIC_train.csv', '/kaggle/input/datasets/veerhanglitch/toxcl-data/SBIC_valid.csv', '/kaggle/input/datasets/veerhanglitch/toxcl-data/SBIC_test.csv', '/kaggle/working/outputs/encode/SBIC/GroNLP-hateBERT', 'encode_SBIC_GroNLP-hateBERT')

In [ ]:
# SBIC BERT

# !python /kaggle/input/datasets/veerhanglitch/toxcl-files/baselines/train_encoder_arch.py \
#     --model_name bert-base-uncased \
#     --train_file /kaggle/input/datasets/veerhanglitch/toxcl-data/SBIC_train.csv \
#     --valid_file /kaggle/input/datasets/veerhanglitch/toxcl-data/SBIC_valid.csv \
#     --test_file /kaggle/input/datasets/veerhanglitch/toxcl-data/SBIC_test.csv \
#     --output_dir /kaggle/working/outputs/encode/SBIC/bert-base-uncased

train_and_upload_safe('bert-base-uncased', '/kaggle/input/datasets/veerhanglitch/toxcl-data/SBIC_train.csv', '/kaggle/input/datasets/veerhanglitch/toxcl-data/SBIC_valid.csv', '/kaggle/input/datasets/veerhanglitch/toxcl-data/SBIC_test.csv', '/kaggle/working/outputs/encode/SBIC/bert-base-uncased', 'encode_SBIC_bert-base-uncased')


In [ ]:
# SBIC ELECTRA

# the cmd stuff was required to avoid a ValueError caused by ELECTRA's non-contiguous attention tensors as they are incompatible with the modern default Safetensors serialization format.
# It fixees the prob by forcing a fallback to the standard PyTorch .bin format.

# !python /kaggle/input/datasets/veerhanglitch/toxcl-files/baselines/train_encoder_arch.py \
#     --model_name google/electra-base-discriminator \
#     --train_file /kaggle/input/datasets/veerhanglitch/toxcl-data/SBIC_train.csv \
#     --valid_file /kaggle/input/datasets/veerhanglitch/toxcl-data/SBIC_valid.csv \
#     --test_file /kaggle/input/datasets/veerhanglitch/toxcl-data/SBIC_valid.csv \
#     --output_dir /kaggle/working/outputs/encode/SBIC/encode_SBIC_google-electra-base-discriminator

patch_logic = (
    "import transformers.utils.versions; "
    "transformers.utils.versions.require_version = lambda *a, **k: None; " # bypas the 'PackageNotFoundError'
    "from transformers import TrainingArguments; "
    "orig = TrainingArguments.__init__; "
    "TrainingArguments.__init__ = lambda self, *a, **k: orig(self, *a, **{**k, 'save_safetensors': False}); " # force .bin save
)

# try except used here also just in case !{cmd} gives an error
try:
    cmd = (
        f"python -c \"{patch_logic} "
        f"import sys; sys.argv = ['', "
        f"'--model_name', 'google/electra-base-discriminator', "
        f"'--train_file', '/kaggle/input/datasets/veerhanglitch/toxcl-data/SBIC_train.csv', "
        f"'--valid_file', '/kaggle/input/datasets/veerhanglitch/toxcl-data/SBIC_valid.csv', "
        f"'--test_file', '/kaggle/input/datasets/veerhanglitch/toxcl-data/SBIC_test.csv', "
        f"'--output_dir', '/kaggle/working/outputs/encode/SBIC/google-electra-base-discriminator']; "
        f"exec(open('/kaggle/input/datasets/veerhanglitch/toxcl-files/baselines/train_encoder_arch.py').read())\""
    )

    !{cmd}

    basline_encoder('google/electra-base-discriminator', '/kaggle/input/datasets/veerhanglitch/toxcl-data/SBIC_train.csv', '/kaggle/input/datasets/veerhanglitch/toxcl-data/SBIC_valid.csv', '/kaggle/input/datasets/veerhanglitch/toxcl-data/SBIC_test.csv', '/kaggle/working/outputs/encode/SBIC/google-electra-base-discriminator', 'encode_SBIC_google-electra-base-discriminator')

except Exception as e:
    print(f"Error: {e}")

finally:
    print("Process complete.")

In [ ]:
# IHC RoBERTa

# !python /kaggle/input/datasets/veerhanglitch/toxcl-files/baselines/train_encoder_arch.py \
#     --model_name roberta-base \
#     --train_file /kaggle/input/datasets/veerhanglitch/toxcl-data/SBIC_train.csv \
#     --valid_file /kaggle/input/datasets/veerhanglitch/toxcl-data/SBIC_valid.csv \
#     --test_file /kaggle/input/datasets/veerhanglitch/toxcl-data/SBIC_test.csv \
#     --output_dir /kaggle/working/outputs/encode/SBIC/roberta-base

basline_encoder('roberta-base', '/kaggle/input/datasets/veerhanglitch/toxcl-data/SBIC_train.csv', '/kaggle/input/datasets/veerhanglitch/toxcl-data/SBIC_valid.csv', '/kaggle/input/datasets/veerhanglitch/toxcl-data/SBIC_test.csv', '/kaggle/working/outputs/encode/SBIC/roberta-base', 'encode_SBIC_roberta-base')

BASELINE 2: Decoder

In [ ]:
# command given in github

# python -m baselines.train_decoder_arch \
#     --model_name_or_path gpt2 \
#     --output_dir {output_dir} \
#     --dataset_name {IHC | SBIC} \
#     --per_device_train_batch_size 8 \
#     --per_device_eval_batch_size 8 \
#     --max_train_steps 20000 \
#     --learning_rate 1e-4 \
#     --text_column {raw_text | text} \
#     --summary_column {explanations | output}

In [ ]:
def basline_decoder(model_path, train_file, valid_file, test_file, output_dir, text_col, summary_col, name):
    zip_path = f"/kaggle/working/{name}_results"
    
    os.makedirs(output_dir, exist_ok=True)
    
    try:
        print(f"Starting Decoder Training: {name}")
        
        !python /kaggle/input/datasets/veerhanglitch/toxcl-files/baselines/train_decoder_arch.py \
            --model_name_or_path {model_path} \
            --train_file {train_file} \
            --validation_file {valid_file} \
            --output_dir {output_dir} \
            --per_device_train_batch_size 8 \
            --per_device_eval_batch_size 8 \
            --max_train_steps 20000 \
            --learning_rate 1e-4 \
            --text_column {text_col} \
            --summary_column {summary_col}
        
        print(f"Zipping results for {name}")
        shutil.make_archive(zip_path, 'zip', output_dir)
        
        run = wandb.init(project="toxcl-decoder-runs", name=name, reinit=True)
        artifact = wandb.Artifact(name=name, type="decoder_output")
        artifact.add_file(f"{zip_path}.zip")
        run.log_artifact(artifact)
        run.finish()
        print(f"SUCCESS: {name} uploaded.")

    except Exception as e:
        print(f"CRASH in {name}: {e}")
        run = wandb.init(project="toxcl-decoder-runs", name=f"FAILED_{name}", reinit=True)
        run.finish()

    finally:
        if os.path.exists(output_dir):
            shutil.rmtree(output_dir)
        if os.path.exists(f"{zip_path}.zip"):
            os.remove(f"{zip_path}.zip")
        print(f"Disk cleared for next run.")

In [ ]:
# IHC raw_text explanations

# !python /kaggle/input/datasets/veerhanglitch/toxcl-files/baselines/train_decoder_arch.py \
#     --model_name_or_path gpt2 \
#     --output_dir /kaggle/working/outputs/decode/IHC/raw/expl
#     --train_file /kaggle/input/datasets/veerhanglitch/toxcl-data/IHC_train.csv \
#     --valid_file /kaggle/input/datasets/veerhanglitch/toxcl-data/IHC_valid.csv \
#     --test_file /kaggle/input/datasets/veerhanglitch/toxcl-data/IHC_valid.csv \
#     --per_device_train_batch_size 8 \
#     --per_device_eval_batch_size 8 \
#     --max_train_steps 20000 \
#     --learning_rate 1e-4 \
#     --text_column raw_text \
#     --summary_column explanations

basline_decoder('gpt2', '/kaggle/input/datasets/veerhanglitch/toxcl-data/IHC_train.csv', '/kaggle/input/datasets/veerhanglitch/toxcl-data/IHC_valid.csv', '/kaggle/input/datasets/veerhanglitch/toxcl-data/IHC_valid.csv', '/kaggle/working/outputs/decode/IHC/raw/expl', 'raw_text', 'explanations', 'dec_IHC_raw_expl')

In [ ]:
# IHC raw_text output

# !python /kaggle/input/datasets/veerhanglitch/toxcl-files/baselines/train_decoder_arch.py \
#     --model_name_or_path gpt2 \
#     --output_dir /kaggle/working/outputs/decode/IHC/raw/out
#     --train_file /kaggle/input/datasets/veerhanglitch/toxcl-data/IHC_train.csv \
#     --valid_file /kaggle/input/datasets/veerhanglitch/toxcl-data/IHC_valid.csv \
#     --test_file /kaggle/input/datasets/veerhanglitch/toxcl-data/IHC_valid.csv \
#     --per_device_train_batch_size 8 \
#     --per_device_eval_batch_size 8 \
#     --max_train_steps 20000 \
#     --learning_rate 1e-4 \
#     --text_column raw_text \
#     --summary_column output

basline_decoder('gpt2', '/kaggle/input/datasets/veerhanglitch/toxcl-data/IHC_train.csv', '/kaggle/input/datasets/veerhanglitch/toxcl-data/IHC_valid.csv', '/kaggle/input/datasets/veerhanglitch/toxcl-data/IHC_valid.csv', '/kaggle/working/outputs/decode/IHC/raw/out', 'raw_text', 'output', 'dec_IHC_raw_out')

In [ ]:
# IHC text explanations

# !python /kaggle/input/datasets/veerhanglitch/toxcl-files/baselines/train_decoder_arch.py \
#     --model_name_or_path gpt2 \
#     --output_dir /kaggle/working/outputs/decode/IHC/txt/expl
#     --train_file /kaggle/input/datasets/veerhanglitch/toxcl-data/IHC_train.csv \
#     --valid_file /kaggle/input/datasets/veerhanglitch/toxcl-data/IHC_valid.csv \
#     --test_file /kaggle/input/datasets/veerhanglitch/toxcl-data/IHC_valid.csv \
#     --per_device_train_batch_size 8 \
#     --per_device_eval_batch_size 8 \
#     --max_train_steps 20000 \
#     --learning_rate 1e-4 \
#     --text_column text \
#     --summary_column explanations

basline_decoder('gpt2', '/kaggle/input/datasets/veerhanglitch/toxcl-data/IHC_train.csv', '/kaggle/input/datasets/veerhanglitch/toxcl-data/IHC_valid.csv', '/kaggle/input/datasets/veerhanglitch/toxcl-data/IHC_valid.csv', '/kaggle/working/outputs/decode/IHC/txt/expl', 'text', 'explanations', 'dec_IHC_txt_expl')

In [ ]:
# IHC text output

# !python /kaggle/input/datasets/veerhanglitch/toxcl-files/baselines/train_decoder_arch.py \
#     --model_name_or_path gpt2 \
#     --output_dir /kaggle/working/outputs/decode/IHC/txt/out
#     --train_file /kaggle/input/datasets/veerhanglitch/toxcl-data/IHC_train.csv \
#     --valid_file /kaggle/input/datasets/veerhanglitch/toxcl-data/IHC_valid.csv \
#     --test_file /kaggle/input/datasets/veerhanglitch/toxcl-data/IHC_valid.csv \
#     --per_device_train_batch_size 8 \
#     --per_device_eval_batch_size 8 \
#     --max_train_steps 20000 \
#     --learning_rate 1e-4 \
#     --text_column text \
#     --summary_column output

basline_decoder('gpt2', '/kaggle/input/datasets/veerhanglitch/toxcl-data/IHC_train.csv', '/kaggle/input/datasets/veerhanglitch/toxcl-data/IHC_valid.csv', '/kaggle/input/datasets/veerhanglitch/toxcl-data/IHC_valid.csv', '/kaggle/working/outputs/decode/IHC/txt/out', 'text', 'output', 'dec_IHC_txt_out')

In [ ]:
# SBIC raw_text explanations

# !python /kaggle/input/datasets/veerhanglitch/toxcl-files/baselines/train_decoder_arch.py \
#     --model_name_or_path gpt2 \
#     --output_dir /kaggle/working/outputs/decode/SBIC/raw/expl
#     --train_file /kaggle/input/datasets/veerhanglitch/toxcl-data/SBIC_train.csv \
#     --valid_file /kaggle/input/datasets/veerhanglitch/toxcl-data/SBIC_valid.csv \
#     --test_file /kaggle/input/datasets/veerhanglitch/toxcl-data/SBIC_test.csv \
#     --per_device_train_batch_size 8 \
#     --per_device_eval_batch_size 8 \
#     --max_train_steps 20000 \
#     --learning_rate 1e-4 \
#     --text_column raw_text \
#     --summary_column explanations

basline_decoder('gpt2', '/kaggle/input/datasets/veerhanglitch/toxcl-data/SBIC_train.csv', '/kaggle/input/datasets/veerhanglitch/toxcl-data/SBIC_valid.csv', '/kaggle/input/datasets/veerhanglitch/toxcl-data/SBIC_test.csv', '/kaggle/working/outputs/decode/SBIC/raw/expl', 'raw_text', 'explanations', 'dec_SBIC_raw_expl')

In [ ]:
# SBIC raw_text output

# !python /kaggle/input/datasets/veerhanglitch/toxcl-files/baselines/train_decoder_arch.py \
#     --model_name_or_path gpt2 \
#     --output_dir /kaggle/working/outputs/decode/SBIC/raw/out
#     --train_file /kaggle/input/datasets/veerhanglitch/toxcl-data/SBIC_train.csv \
#     --valid_file /kaggle/input/datasets/veerhanglitch/toxcl-data/SBIC_valid.csv \
#     --test_file /kaggle/input/datasets/veerhanglitch/toxcl-data/SBIC_test.csv \
#     --per_device_train_batch_size 8 \
#     --per_device_eval_batch_size 8 \
#     --max_train_steps 20000 \
#     --learning_rate 1e-4 \
#     --text_column raw_text \
#     --summary_column output

basline_decoder('gpt2', '/kaggle/input/datasets/veerhanglitch/toxcl-data/SBIC_train.csv', '/kaggle/input/datasets/veerhanglitch/toxcl-data/SBIC_valid.csv', '/kaggle/input/datasets/veerhanglitch/toxcl-data/SBIC_test.csv', '/kaggle/working/outputs/decode/SBIC/raw/out', 'raw_text', 'output', 'dec_SBIC_raw_out')

In [ ]:
# SBIC text explanations

# !python /kaggle/input/datasets/veerhanglitch/toxcl-files/baselines/train_decoder_arch.py \
#     --model_name_or_path gpt2 \
#     --output_dir /kaggle/working/outputs/decode/SBIC/txt/expl
#     --train_file /kaggle/input/datasets/veerhanglitch/toxcl-data/SBIC_train.csv \
#     --valid_file /kaggle/input/datasets/veerhanglitch/toxcl-data/SBIC_valid.csv \
#     --test_file /kaggle/input/datasets/veerhanglitch/toxcl-data/SBIC_test.csv \
#     --per_device_train_batch_size 8 \
#     --per_device_eval_batch_size 8 \
#     --max_train_steps 20000 \
#     --learning_rate 1e-4 \
#     --text_column text \
#     --summary_column explanations

basline_decoder('gpt2', '/kaggle/input/datasets/veerhanglitch/toxcl-data/SBIC_train.csv', '/kaggle/input/datasets/veerhanglitch/toxcl-data/SBIC_valid.csv', '/kaggle/input/datasets/veerhanglitch/toxcl-data/SBIC_test.csv', '/kaggle/working/outputs/decode/SBIC/txt/expl', 'text', 'explanations', 'dec_SBIC_txt_expl')

In [ ]:
# SBIC text output

# !python /kaggle/input/datasets/veerhanglitch/toxcl-files/baselines/train_decoder_arch.py \
#     --model_name_or_path gpt2 \
#     --output_dir /kaggle/working/outputs/decode/SBIC/txt/out
#     --train_file /kaggle/input/datasets/veerhanglitch/toxcl-data/SBIC_train.csv \
#     --valid_file /kaggle/input/datasets/veerhanglitch/toxcl-data/SBIC_valid.csv \
#     --test_file /kaggle/input/datasets/veerhanglitch/toxcl-data/SBIC_test.csv \
#     --per_device_train_batch_size 8 \
#     --per_device_eval_batch_size 8 \
#     --max_train_steps 20000 \
#     --learning_rate 1e-4 \
#     --text_column text \
#     --summary_column output

basline_decoder('gpt2', '/kaggle/input/datasets/veerhanglitch/toxcl-data/SBIC_train.csv', '/kaggle/input/datasets/veerhanglitch/toxcl-data/SBIC_valid.csv', '/kaggle/input/datasets/veerhanglitch/toxcl-data/SBIC_test.csv', '/kaggle/working/outputs/decode/SBIC/txt/out', 'text', 'output', 'dec_SBIC_txt_out')